# Breast Cancer Diagnostic Classifier — Training Notebook (Google Colab)

Classifies breast masses as **malignant / benign** from 30 cell-nucleus measurements (mean, SE, worst). Dataset: **Breast Cancer Wisconsin (Diagnostic)** — UCI, built into `sklearn.datasets` so no download is needed.

**Note:** proposed own-project per course rules — needs instructor approval.

**Steps:** install deps → train → evaluate → download `model.joblib` → push repo → deploy on Streamlit Cloud.

## 1. Install dependencies

In [ ]:
!pip install -q pandas scikit-learn joblib numpy streamlit

## 2. Train the model

Random Forest on the 30 numeric features. Target relabelled so **1 = malignant** (sklearn's raw target has 0 = malignant).

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
feature_names = list(data.feature_names)
X = data.data
y = 1 - data.target  # 1 = malignant

print(f'Loaded {X.shape[0]} samples x {X.shape[1]} features')
print(f'Malignant rate: {y.mean():.1%}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

clf = RandomForestClassifier(
    n_estimators=800, min_samples_leaf=1,
    n_jobs=-1, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
print(f'\nAccuracy : {acc:.4f}')
print(f'ROC-AUC  : {auc:.4f}')
print(classification_report(y_test, y_pred))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))

perm = permutation_importance(clf, X_test, y_test, n_repeats=10,
                             random_state=42, n_jobs=-1, scoring='roc_auc')
order = np.argsort(perm.importances_mean)[::-1][:15]
importances = {feature_names[i]: round(float(perm.importances_mean[i]), 4)
               for i in order}
print('\nTop-15 features:')
for k, v in importances.items():
    print(f'  {k}: {v}')

## 3. Save `model.joblib` and download it

In [ ]:
import json

report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
cm = confusion_matrix(y_test, y_pred).tolist()

bundle = {
    'model': clf,
    'feature_names': feature_names,
    'feature_ranges': {
        name: [float(X[:, i].min()), float(X[:, i].max())]
        for i, name in enumerate(feature_names)},
    'feature_importances': importances,
    'metrics': {
        'accuracy': round(float(acc), 4),
        'roc_auc': round(float(auc), 4),
        'classification_report': report,
        'confusion_matrix': cm,
        'train_rows': int(len(X_train)),
        'test_rows': int(len(X_test)),
    },
}
joblib.dump(bundle, 'model.joblib', compress=3)

with open('metrics.json', 'w') as f:
    json.dump(bundle['metrics'], f, indent=2)

from google.colab import files
files.download('model.joblib')
files.download('metrics.json')